In [1]:
# Import required python modules
import os
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pandas as pd
import sqlite3
from astropy.coordinates import SkyCoord
import astropy.units as u
import rubin_scheduler.scheduler.utils as sched_utils
from rubin_scheduler.utils import ddf_locations, ddf_locations_skycoord
from rubin_sim.data import get_baseline
import rubin_sim.maf as maf
from scipy.stats import binned_statistic

from IPython.display import display, Markdown

In [2]:
# v4.3.1 runs
summaries = maf.get_metric_summaries(summary_source='summary.h5')
print(f"This summary h5 file contains information on {len(summaries.index)} simulations.")
print(summaries.index)

This summary h5 file contains information on 58 simulations.
Index(['all_early_v4.3.2_10yrs', 'baseline_v2.0_10yrs', 'baseline_v2.1_10yrs',
       'baseline_v2.2_10yrs', 'baseline_v3.0_10yrs', 'baseline_v3.2_10yrs',
       'baseline_v3.3_10yrs', 'baseline_v3.4_10yrs', 'baseline_v3.5_10yrs',
       'baseline_v3.6_10yrs', 'baseline_v4.0_10yrs', 'baseline_v4.1_10yrs',
       'baseline_v4.2_10yrs', 'baseline_v4.3.1_10yrs', 'baseline_v4.3.2_10yrs',
       'ddf_accor_skipizy_v4.3.2_10yrs', 'ddf_accor_v4.3.2_10yrs',
       'ddf_roll__1_v4.3.1_10yrs', 'ddf_roll_baseline_v4.3.2_10yrs',
       'desc_ddf_gen_0.70_co_v4.3.1_10yrs',
       'desc_ddf_gen_0.70_sn_v4.3.1_10yrs', 'desc_ddf_gen_0.70_v4.3.1_10yrs',
       'desc_ddf_gen_0.70_wz_v4.3.1_10yrs',
       'desc_ddf_gen_0.75_co_v4.3.1_10yrs',
       'desc_ddf_gen_0.75_sn_v4.3.1_10yrs',
       'desc_ddf_gen_0.75_wz_v4.3.1_10yrs',
       'desc_ddf_gen_0.80_co_v4.3.1_10yrs',
       'desc_ddf_gen_0.80_sn_v4.3.1_10yrs', 'desc_ddf_gen_0.80_v4.3.1_10yr

In [3]:
metric_subsets = maf.get_metric_subsets('metric_subsets.json')
msets = list(metric_subsets.groupby('metric subset').first().index)

m5_SRD_design = {'u': 23.90, 'g': 25.00, 'r': 24.70, 'i': 24.00, 'z': 23.30, 'y': 22.10}
m5_SRD_minimum = {'u': 23.40, 'g':24.60, 'r': 24.30, 'i': 23.60, 'z': 22.90, 'y': 21.70}

# Weather variations on baseline strategy
weather_runs = [r for r in summaries.index if 'weather' in r]
offsets = [int(w.split('dso')[-1].split('v4')[0]) for w in weather_runs]
idx = np.argsort(offsets)
weather_runs = [weather_runs[i] for i in idx]
print("weather runs", len(weather_runs))

# Start date variations
start_dates = [r for r in summaries.index if 'start_date' in r]
offsets = [int(r.split('mjdp')[-1].split('_')[0]) for r in start_dates]
idx = np.argsort(offsets)
start_dates = [start_dates[i] for i in idx]
print("start_dates", len(start_dates))


# Baseline sims, and put them in order of evolution

baseline_dict = {'retro_baseline_v2.0_10yrs': 'v1.x', 
                 'baseline_v2.0_10yrs':'v2.0', 
                 'baseline_v2.1_10yrs':'v2.1',
                 'baseline_v2.2_10yrs':'v2.2',
                 'baseline_v3.0_10yrs':'v3.0',
                 'baseline_v3.2_10yrs':'v3.2',
                'baseline_v3.3_10yrs':'v3.3',
                'baseline_v3.4_10yrs': 'v3.4',
                'baseline_v3.5_10yrs': 'v3.5', 
                 'baseline_v3.6_10yrs': 'v3.6',
                 'baseline_v4.0_10yrs': 'v4.0',
                 'baseline_v4.1_10yrs': 'v4.1',
                 'baseline_v4.2_10yrs': 'v4.2'
                }

newbaselines_dict = {'baseline_v4.3.1_10yrs': "baseline v4.3.1",
                     'four_roll_v4.3.1_10yrs': "four cycle v4.3.1",
                    'one_snap_v4.3.1_10yrs' : "single snap v4.3.1",
                    }

rdict = {}
for k in baseline_dict:
    rdict[k] = baseline_dict[k]
for k in newbaselines_dict:
    rdict[k] = newbaselines_dict[k]

runs = list(rdict.keys())
baselinerun = 'baseline_v4.3.1_10yrs'
baseline = baselinerun

ddf_desc = [ "desc_ddf_gen_0.70_sn_v4.3.1_10yrs",
            "desc_ddf_gen_0.75_sn_v4.3.1_10yrs",
            "desc_ddf_gen_0.80_sn_v4.3.1_10yrs",
            "desc_ddf_gen_0.70_co_v4.3.1_10yrs",
            "desc_ddf_gen_0.75_co_v4.3.1_10yrs",
            "desc_ddf_gen_0.80_co_v4.3.1_10yrs",           
            "desc_ddf_gen_0.70_wz_v4.3.1_10yrs",
            "desc_ddf_gen_0.75_wz_v4.3.1_10yrs",
            "desc_ddf_gen_0.80_wz_v4.3.1_10yrs",
            "ddf_roll__1_v4.3.1_10yrs",
           ]
ddf_accord = ["ddf_accor_skipizy_v4.3.2_10yrs", "ddf_accor_v4.3.2_10yrs",]
ddf_roll = ["ddf_roll_baseline_v4.3.2_10yrs"]

ddf_sets = [ddf_desc, ddf_accord, ddf_roll]
ddf_runs = [r for r in ddf_desc] + [r for r in ddf_accord] + [r for r in ddf_roll]

outdir = 'tmp_fig'
try:
    os.mkdir(outdir)
except:
    pass

weather runs 14
start_dates 4


In [4]:
all_runs =  ['baseline_v4.3.2_10yrs', 'baseline_v4.3.1_10yrs'] + ddf_runs
all_runs

['baseline_v4.3.2_10yrs',
 'baseline_v4.3.1_10yrs',
 'desc_ddf_gen_0.70_sn_v4.3.1_10yrs',
 'desc_ddf_gen_0.75_sn_v4.3.1_10yrs',
 'desc_ddf_gen_0.80_sn_v4.3.1_10yrs',
 'desc_ddf_gen_0.70_co_v4.3.1_10yrs',
 'desc_ddf_gen_0.75_co_v4.3.1_10yrs',
 'desc_ddf_gen_0.80_co_v4.3.1_10yrs',
 'desc_ddf_gen_0.70_wz_v4.3.1_10yrs',
 'desc_ddf_gen_0.75_wz_v4.3.1_10yrs',
 'desc_ddf_gen_0.80_wz_v4.3.1_10yrs',
 'ddf_roll__1_v4.3.1_10yrs',
 'ddf_accor_skipizy_v4.3.2_10yrs',
 'ddf_accor_v4.3.2_10yrs',
 'ddf_roll_baseline_v4.3.2_10yrs']

In [5]:
conn = sqlite3.connect('baseline_v4.3.2_10yrs.db')
query = 'select night, moonPhase from observations where night<100'
vals = pd.read_sql(query, conn)
nstart = vals.query('moonPhase < 50').night.min()
print(nstart)

11


In [6]:
# Basic DDF information, number of visits from baseline simulation
ddfs = ddf_locations_skycoord()
nvis = {}
frac_seeing = {}
u_per_30 = {}
u_per_30 = {}
max_u_per_30 = {}
pair_count = {}
for run in all_runs:
    conn = sqlite3.connect(run + ".db") 
    nvis[run] = {}
    frac_seeing[run] = {}
    u_per_30[run] = {}
    max_u_per_30[run] = {}
    pair_count[run] = {}
    for i in ddfs:
        query = f"select count(*) from observations where scheduler_note like '%{i}%'"
        nvis[run][i] = int(pd.read_sql(query, conn).values[0][0])
        query = f"select seeingFwhmEff from observations where scheduler_note like '%{i}%' and band == 'r'"
        seeing = pd.read_sql(query, conn)['seeingFwhmEff'].values
        frac_seeing[run][i] = len(np.where(seeing < 1.2)[0]) / len(seeing)
        query = f"select observationStartMJD, night, band, moonPhase from observations where scheduler_note like '%{i}%'"
        vals = pd.read_sql(query, conn)
        # Find times with 60 u band visits in 30 nights
        counts = vals.query("band == 'u'").groupby('night').count()['observationStartMJD']
        visits_per_30, bin_edges, bin_count = binned_statistic(counts.index, counts.values, bins=np.arange(nstart, 3652, 30), statistic='sum')
        max_u_per_30[run][i] = max(visits_per_30)
        u_per_30[run][i] = len(np.where(visits_per_30 > 60)[0])
        # Find pairs of nights with the required number of visits per pair-night
        t = vals.groupby(['night', 'band']).agg({'band': 'count'}).rename(columns={'band': 'count'}).reset_index().pivot(index=['night'], columns=['band']).fillna(0)
        t = t.droplevel(level=0, axis=1)
        t = t.drop(labels='u', axis=1)
        t['total'] = t.sum(axis=1)
        t = t.query('total > 0')
        s = t.query('g>1 and r>1 and i>3 and z>5 and y>4')
        # find and drop all the impossible y combinations
        def find_drops(df, column, minimum_val):
            idx_bad = np.where(df[column] < minimum_val)[0]
            idx_good = np.where(df[column] >= minimum_val)[0]
            drop = []
            for i in idx_bad:
                if (i+1 not in idx_good) and (i-1 not in idx_good):
                    drop.append(i)
            return drop
        drop = find_drops(t, 'y', 4)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'z', 5)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'i', 3)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'r', 1)
        t = t.drop(index=t.iloc[drop].index)
        drop = find_drops(t, 'g', 1)
        t = t.drop(index=t.iloc[drop].index)
        nights = t.index.values
        pairs = np.where(np.diff(nights) < 2)[0]
        pnights = t.iloc[np.sort(np.concatenate([pairs, pairs+1]))].index.values
        double_count = [p for p in pnights if p in s.index.values]
        pair_count[run][i] = len(pairs) + len(s) - len(double_count)
        
pdfs = {}
for i in ddfs:
    ra = ddfs[i].ra.deg
    dec = ddfs[i].dec.deg
    coord = ddfs[i]
    ra_hours = coord.ra.hms
    eclip_lat = coord.barycentrictrueecliptic.lat.deg
    eclip_lon = coord.barycentrictrueecliptic.lon.deg
    gal_lon = coord.galactic.l.deg
    gal_lat = coord.galactic.b.deg
    pdfs[i] = [ra, dec, gal_lon, gal_lat, eclip_lon, eclip_lat]
d = pd.DataFrame(pdfs, index=['RA', 'Dec', 'Gal l', 'Gal b', 'Eclip l', 'Eclip b']).round(2)
nvisits = pd.DataFrame(nvis).T
seeing = pd.DataFrame(frac_seeing).T
max_u_per_30 = pd.DataFrame(max_u_per_30).T
u_per_30 = pd.DataFrame(u_per_30).T
night_pairs = pd.DataFrame(pair_count).T

In [9]:
display(Markdown("Basic DDF information"))
display(d)

display(Markdown("Number of visits per DDF"))
display(nvisits)

display(Markdown("Fraction of DDF r-band visits with seeing < 1.2\""))
display(seeing.round(2))

display(Markdown("Maximum number of u band visits within 30 nights"))
display(max_u_per_30)

display(Markdown("Number of months with >60 u band visits in 30 nights"))
display(u_per_30)

display(Markdown("Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits"))
display(night_pairs)

Basic DDF information

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
RA,9.45,35.57,52.98,150.11,58.90,63.60
Dec,-44.02,-4.82,-28.12,2.23,-49.32,-47.60
Gal l,311.29,171.10,224.07,236.78,257.90,254.48
Gal b,-72.88,-58.91,-54.60,42.13,-48.46,-45.77
Eclip l,346.66,31.59,40.81,151.39,32.00,40.97
Eclip b,-43.20,-17.92,-45.44,-9.34,-66.61,-66.60


Number of visits per DDF

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v4.3.2_10yrs,22758,23287,23728,45695,11829,11794
baseline_v4.3.1_10yrs,22769,23330,23712,45649,11824,11779
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,6865,52956,7033,56400,3496,3466
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,6342,50354,6494,52466,3290,3246
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,8189,48051,8365,51165,4208,4149
desc_ddf_gen_0.70_co_v4.3.1_10yrs,10530,49256,10598,52791,5278,5233
desc_ddf_gen_0.75_co_v4.3.1_10yrs,10029,47903,10074,50004,5058,5003
desc_ddf_gen_0.80_co_v4.3.1_10yrs,12125,46616,12274,49731,6065,6006
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,15288,46513,15309,50402,7589,7540
desc_ddf_gen_0.75_wz_v4.3.1_10yrs,15264,46093,15281,48377,7604,7544


Fraction of DDF r-band visits with seeing < 1.2"

...

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v4.3.2_10yrs,24.0,24.0,24.0,120.0,12.0,12.0
baseline_v4.3.1_10yrs,24.0,24.0,24.0,120.0,12.0,12.0
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,76.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,76.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,76.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.70_co_v4.3.1_10yrs,77.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.75_co_v4.3.1_10yrs,77.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.80_co_v4.3.1_10yrs,84.0,95.0,95.0,133.0,38.0,38.0
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,112.0,114.0,96.0,176.0,48.0,48.0
desc_ddf_gen_0.75_wz_v4.3.1_10yrs,112.0,112.0,96.0,176.0,48.0,48.0


Number of months with >60 u band visits in 30 nights

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v4.3.2_10yrs,0,0,0,12,0,0
baseline_v4.3.1_10yrs,0,0,0,12,0,0
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,2,2,3,8,0,0
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,2,2,3,8,0,0
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,3,3,3,15,0,0
desc_ddf_gen_0.70_co_v4.3.1_10yrs,4,17,5,28,0,0
desc_ddf_gen_0.75_co_v4.3.1_10yrs,4,17,4,26,0,0
desc_ddf_gen_0.80_co_v4.3.1_10yrs,4,15,5,26,0,0
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,43,48,40,50,0,0
desc_ddf_gen_0.75_wz_v4.3.1_10yrs,42,48,40,51,0,0


Number of 48-hour intervals with g>1, r>1, i>3, z>5, y>4 visits

,ELAISS1,XMM_LSS,ECDFS,COSMOS,EDFS_a,EDFS_b
baseline_v4.3.2_10yrs,164,153,169,156,162,162
baseline_v4.3.1_10yrs,164,155,169,156,161,162
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,1,113,1,91,0,0
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,1,90,1,72,0,0
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,1,60,1,51,0,0
desc_ddf_gen_0.70_co_v4.3.1_10yrs,11,127,13,101,0,0
desc_ddf_gen_0.75_co_v4.3.1_10yrs,11,103,13,88,0,0
desc_ddf_gen_0.80_co_v4.3.1_10yrs,11,77,12,66,0,0
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,116,187,112,159,25,23
desc_ddf_gen_0.75_wz_v4.3.1_10yrs,114,184,110,152,24,22


In [8]:
for i in ddfs:
    if i == 'EDFS_a':
        name = 'EDFS'
    elif i == "EDFS_b":
        continue
    else:
        name = i
    msub = metric_subsets.loc['DDF Depths'].query("metric.str.contains(@name) and metric.str.contains('Coadd')")
    print(name)
    display(summaries.loc[all_runs, msub['metric']].round(2).rename(columns=msub['short_name']))

ELAISS1


metric,CoaddedM5 ELAISS1 u,CoaddedM5 ELAISS1 g,CoaddedM5 ELAISS1 r,CoaddedM5 ELAISS1 i,CoaddedM5 ELAISS1 z,CoaddedM5 ELAISS1 y
run,,,,,,
baseline_v4.3.2_10yrs,26.86,28.35,28.34,27.97,27.45,26.12
baseline_v4.3.1_10yrs,26.85,28.34,28.33,27.98,27.46,26.13
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,27.27,27.69,27.70,27.28,26.69,25.66
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,27.27,27.68,27.47,27.26,26.67,25.65
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,27.38,27.68,27.70,27.43,26.67,25.77
desc_ddf_gen_0.70_co_v4.3.1_10yrs,27.48,27.98,27.87,27.46,26.85,25.88
desc_ddf_gen_0.75_co_v4.3.1_10yrs,27.47,27.97,27.70,27.45,26.85,25.87
desc_ddf_gen_0.80_co_v4.3.1_10yrs,27.51,27.97,27.85,27.57,26.96,25.96
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,27.65,27.98,27.98,27.69,27.08,26.10


XMM_LSS


metric,CoaddedM5 XMM_LSS u,CoaddedM5 XMM_LSS g,CoaddedM5 XMM_LSS r,CoaddedM5 XMM_LSS i,CoaddedM5 XMM_LSS z,CoaddedM5 XMM_LSS y
run,,,,,,
baseline_v4.3.2_10yrs,26.75,28.25,28.26,27.87,27.37,26.06
baseline_v4.3.1_10yrs,26.75,28.26,28.25,27.87,27.37,26.06
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,27.25,28.16,28.47,28.74,28.10,26.10
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,27.21,28.09,28.32,28.77,28.05,26.39
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,27.32,28.01,28.20,28.78,28.02,26.38
desc_ddf_gen_0.70_co_v4.3.1_10yrs,27.47,28.21,28.49,28.65,28.02,26.08
desc_ddf_gen_0.75_co_v4.3.1_10yrs,27.46,28.17,28.36,28.69,28.02,26.26
desc_ddf_gen_0.80_co_v4.3.1_10yrs,27.46,28.14,28.28,28.72,28.00,26.39
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,27.63,28.23,28.51,28.57,27.97,26.08


ECDFS


metric,CoaddedM5 ECDFS u,CoaddedM5 ECDFS g,CoaddedM5 ECDFS r,CoaddedM5 ECDFS i,CoaddedM5 ECDFS z,CoaddedM5 ECDFS y
run,,,,,,
baseline_v4.3.2_10yrs,26.94,28.40,28.42,28.03,27.52,26.20
baseline_v4.3.1_10yrs,26.93,28.40,28.41,28.04,27.52,26.19
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,27.39,27.79,27.78,27.37,26.77,25.77
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,27.40,27.80,27.57,27.36,26.78,25.75
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,27.49,27.79,27.79,27.54,26.76,25.87
desc_ddf_gen_0.70_co_v4.3.1_10yrs,27.56,28.05,27.93,27.52,26.93,25.98
desc_ddf_gen_0.75_co_v4.3.1_10yrs,27.56,28.04,27.77,27.52,26.93,25.97
desc_ddf_gen_0.80_co_v4.3.1_10yrs,27.61,28.06,27.94,27.65,27.06,26.06
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,27.73,28.05,28.05,27.75,27.16,26.19


COSMOS


metric,CoaddedM5 COSMOS u,CoaddedM5 COSMOS g,CoaddedM5 COSMOS r,CoaddedM5 COSMOS i,CoaddedM5 COSMOS z,CoaddedM5 COSMOS y
run,,,,,,
baseline_v4.3.2_10yrs,27.21,28.65,28.66,28.27,27.76,26.33
baseline_v4.3.1_10yrs,27.19,28.65,28.66,28.27,27.77,26.33
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,27.24,28.14,28.45,28.72,28.07,26.03
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,27.20,28.05,28.29,28.72,28.00,26.32
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,27.32,27.95,28.16,28.71,27.96,26.32
desc_ddf_gen_0.70_co_v4.3.1_10yrs,27.44,28.19,28.47,28.63,28.00,26.02
desc_ddf_gen_0.75_co_v4.3.1_10yrs,27.44,28.14,28.34,28.64,27.98,26.18
desc_ddf_gen_0.80_co_v4.3.1_10yrs,27.45,28.07,28.23,28.65,27.93,26.30
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,27.61,28.20,28.50,28.55,27.95,26.01


EDFS


metric,CoaddedM5 EDFS u,CoaddedM5 EDFS g,CoaddedM5 EDFS r,CoaddedM5 EDFS i,CoaddedM5 EDFS z,CoaddedM5 EDFS y
run,,,,,,
baseline_v4.3.2_10yrs,26.67,28.04,28.10,27.69,27.20,25.86
baseline_v4.3.1_10yrs,26.66,28.05,28.09,27.69,27.20,25.86
desc_ddf_gen_0.70_sn_v4.3.1_10yrs,26.97,27.53,27.54,27.09,26.48,25.48
desc_ddf_gen_0.75_sn_v4.3.1_10yrs,26.98,27.54,27.37,27.08,26.49,25.48
desc_ddf_gen_0.80_sn_v4.3.1_10yrs,27.09,27.51,27.52,27.23,26.47,25.56
desc_ddf_gen_0.70_co_v4.3.1_10yrs,27.15,27.73,27.63,27.21,26.62,25.65
desc_ddf_gen_0.75_co_v4.3.1_10yrs,27.15,27.72,27.51,27.20,26.61,25.65
desc_ddf_gen_0.80_co_v4.3.1_10yrs,27.20,27.73,27.62,27.30,26.72,25.72
desc_ddf_gen_0.70_wz_v4.3.1_10yrs,27.31,27.73,27.72,27.40,26.83,25.87
